# PEFT Adapter Parameter Footprint Profiles

This notebook calculates and compares the trainable parameters size of Prompt Tuning, Prefix Tuning, and LoRA.

In [1]:
import torch
import torch.nn as nn

class AdapterComparison:
    def __init__(self, in_features=512, out_features=512, prompt_len=10, prefix_len=10):
        # 1. Base Linear Layer (Frozen)
        self.base = nn.Linear(in_features, out_features)
        
        # 2. Prompt Tuning Parameter size
        self.prompt_embeddings = nn.Parameter(torch.randn(prompt_len, in_features))
        
        # 3. Prefix Tuning Keys/Values size (across 12 layers)
        self.prefix_key_values = nn.Parameter(torch.randn(12, 2, prefix_len, 8, in_features // 8))
        
        # 4. LoRA Adapter size (r=8)
        self.lora_A = nn.Parameter(torch.randn(8, in_features))
        self.lora_B = nn.Parameter(torch.randn(out_features, 8))
        
    def profile_sizes(self):
        prompt_params = self.prompt_embeddings.numel()
        prefix_params = self.prefix_key_values.numel()
        lora_params = self.lora_A.numel() + self.lora_B.numel()
        base_params = self.base.weight.numel() + self.base.bias.numel()
        
        print(f"Base Layer Parameters: {base_params:,}")
        print(f"Prompt Tuning Parameters: {prompt_params:,} ({prompt_params/base_params*100:.3f}% of base)")
        print(f"Prefix Tuning Parameters: {prefix_params:,} ({prefix_params/base_params*100:.3f}% of base)")
        print(f"LoRA (r=8) Parameters: {lora_params:,} ({lora_params/base_params*100:.3f}% of base)")

profile = AdapterComparison()
profile.profile_sizes()


Base Layer Parameters: 262,656
Prompt Tuning Parameters: 5,120 (1.949% of base)
Prefix Tuning Parameters: 122,880 (46.784% of base)
LoRA (r=8) Parameters: 8,192 (3.119% of base)


### Output Explanation
The output logs present parameter calculations. For example, a rank 8 LoRA represents only $\sim 3\%$ parameter overhead compared to the base weight dimension.